In [1]:
import collections
import tests.eval as eval
import faiss
import os
import re
import retrievers
import torch
from transformers import AutoTokenizer, AutoModel
from tqdm import tqdm
import utils

In [2]:
# Variables

TITLES = '.titles.pkl'
REDIRECTS = '.redirects.pkl'
CORPUS = '.wiki'
FAISS = '.index.faiss'
QUESTIONS = 'assets/questions.txt'

MODEL_NAME = 'sentence-transformers/all-MiniLM-L6-v2'
BATCH_SIZE = 128  # Adjust based on VRAM (GTX 1070 has 8GB)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModel.from_pretrained(MODEL_NAME).to(DEVICE)
model.eval()

# FAISS
# Dim 384 for MiniLM-L6. FlatL2 is simple and accurate for this scale.
dimension = 384
index = faiss.IndexFlatL2(dimension)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [3]:
# Load Corpus
    
wiki_titles = []
wiki_redirects = collections.defaultdict(lambda: [])
wiki_content = []
num_pages = 0

for title, content in utils.yield_wiki_corpus(CORPUS):
    num_pages += 1
    content = utils.clean_mediawiki(content)
    
    # if alias, track but dont actually index
    found = re.search(r'^\s*#redirect\s+(.*?)$', content, re.IGNORECASE|re.MULTILINE)
    if found:
        real_page = found.group(1).strip()
        wiki_redirects[real_page].append(title)
        continue
    
    # subdivide into chunks
    for chunk in utils.chunk_text(title, content, window_size=100, overlap=20):
        wiki_titles.append(title)
        wiki_content.append(chunk)

Reading:  .wiki/enwiki-20140602-pages-articles.xml-1259.txt


In [4]:
# Print some stats for verification

print('Num Chunks:', len(wiki_titles),'==',len(wiki_content))
print('Num Pages (include redirects):',num_pages)
print('Num Pages (no redirects):',len(set(wiki_titles)))
print('Num Redirects:',sum(len(v) for v in wiki_redirects.values()))

Num Chunks: 1400380 == 1400380
Num Pages (include redirects): 279758
Num Pages (no redirects): 137229
Num Redirects: 142481


In [5]:
# Pickle the mapping: docID -> title
#                     title -> aliases
    
utils.save_pickle(wiki_titles, TITLES)
utils.save_pickle(dict(wiki_redirects), REDIRECTS)
print(f"Success saving {TITLES}, {REDIRECTS}")

Success saving .titles.pkl, .redirects.pkl


In [6]:
# Test retrieve from pickle

reloaded_titles = utils.load_pickle(TITLES)
reloaded_redirects = utils.load_pickle(REDIRECTS)
len_titles = len(reloaded_titles)

print('First Entry (0):',reloaded_titles[0])
print(f'Last Entry ({len_titles-1}):',reloaded_titles[len_titles-1], end='\n\n')

print("(First few redirects)\n")
i = 0
for k,v in sorted(reloaded_redirects.items()):
    # if len(v) > 1:
    print(k,'->',v)
    i += 1
    if i > 5: break

First Entry (0): BBS
Last Entry (1400379): Google Test

(First few redirects)

!Kung language -> ['!Xũ', 'Kung language', '!Ku language', '!Hu language', 'Qxü language', '!Xun language', '!Khung language']
"Crocodile" Dundee -> ['Crock dundee']
"Crocodile"_Dundee_II -> ["'Crocodile' Dundee II"]
"Freeway" Rick Ross -> ['Ricky D. Ross']
"Good Hair" and Other Dubious Distinctions -> ['"Good Hair" and other Dubious Distinctions']
"The Spaghetti Incident?" -> ['TSI?']


In [7]:
# Batch Encoding

if os.path.exists(FAISS):
    
    print(f'FAISS index {FAISS} already exists from a previous run.')
    print('\tSkipping creation, as this step is expensive.')
    print('\tTo regenerate index, please remove or rename existing.')
    
else:
    
    def mean_pooling(model_output, attention_mask):
        """Perform mean pooling on token embeddings to get a single vector."""
        token_embeddings = model_output[0]
        input_mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
        return torch.sum(token_embeddings * input_mask_expanded, 1) / torch.clamp(input_mask_expanded.sum(1), min=1e-9)


    for i in tqdm(range(0, len(wiki_content), BATCH_SIZE)):
        batch_text = wiki_content[i : i + BATCH_SIZE]
        
        # Tokenization
        encoded_input = tokenizer(
            batch_text, 
            padding=True, 
            truncation=True, 
            max_length=128, 
            return_tensors='pt'
        ).to(DEVICE)

        with torch.no_grad():
            # Use Automatic Mixed Precision (FP16) for speed boost on GTX 1070
            with torch.cuda.amp.autocast():
                model_output = model(**encoded_input)
                
            # Pool embeddings and move to CPU
            batch_embeddings = mean_pooling(model_output, encoded_input['attention_mask'])
            batch_embeddings = batch_embeddings.cpu().numpy()
            
            # Add to FAISS index immediately to save system RAM
            index.add(batch_embeddings.astype('float32'))

    # Save the index
    faiss.write_index(index, FAISS)
    print(f"Success saving {FAISS}")

FAISS index .index.faiss already exists from a previous run.
	Skipping creation, as this step is expensive.
	To regenerate index, please remove or rename existing.


In [8]:
# Evaluation

ir = retrievers.JeopardyDPR(FAISS, TITLES, REDIRECTS, tokenizer, model)
questions = [_ for _ in utils.yield_questions(QUESTIONS)]
k = 100

eval.eval_mrr(ir, questions, k)
eval.eval_p_at_1(ir, questions, k)

MRR@100 (JeopardyDPR): 0.0
P@1 (JeopardyDPR): 0.0
